# Capstone Project : Financial DNA
#### <i> A Client Profiling and Fraud Detection System for Fintech Lending

## Phase 1 : Business Understanding
### 1. Background
Fintech lenders make decisions about loan limits and client risk using data that already sits in their systems, but this data is rarely used to its full potential. Client information such as loan amounts, balances, repayment history, arrears, account status, and demographics is routinely collected as part of normal lending operations, yet it is typically used only to process the loan in front of the officer rather than to build a fuller picture of the client over time.
This project, Financial DNA, is proposed as a capstone system that draws on this existing loan and repayment history to build a behavioural profile for every client described as a behavioural fingerprint derived from their loan history, repayment patterns, arrears, and balances. The intent is to turn data a lender already holds into a working system, rather than to introduce a requirement for new data collection.


### 2. Problem Statement
Two specific problems motivate the project:
•	Fragmented risk assessment: client risk is often assessed loan by loan rather than as a full behavioural picture of the client over time, meaning a lender's view of a client can look different from one application to the next instead of reflecting a consistent history.
•	Reactive fraud detection: fraud is usually detected after it happens, once losses are already incurred, rather than flagged as it develops, meaning the lender is typically reacting to a loss rather than catching it in progress.
Both problems are treated as a single underlying gap rather than two unrelated issues: the absence of one unified system, built from a client's loan and repayment history, that can address risk assessment and fraud detection together.


### 3. Business Objectives
#### 3.1 Business Objectives
The core idea is to build a Financial DNA profile for every client. This profile is designed to do two things at once, and these form the project's two business objectives:
•	Client Understanding: classify each client into a behavioural segment (for example, consistent payer, seasonal borrower, or high-risk repeat defaulter) and recommend an appropriate loan limit based on that profile.
•	Fraud Detection: continuously check whether a client's current behaviour still matches their own historical DNA, or matches known fraud patterns, and flag accounts that drift from expected behaviour.
Because both objectives are built from the same underlying client features, they are to be delivered as a single product, sharing one data pipeline, rather than as two separate systems. A further objective sits alongside these two: deployment is treated as mandatory rather than optional, meaning the finished models must be saved and served through a REST API, with a lightweight web UI on top, so that a loan officer can enter or select a client and instantly see their Financial DNA profile, recommended loan limit, and current fraud risk flag.

#### 3.2 Business Success Criteria
By the end of the capstone period, the project is considered successful from a business perspective if it delivers a deployed, demo-ready system that turns raw loan and repayment history into three practical outputs for a lending business: a behavioural profile per client, a data-driven loan limit recommendation, and a real-time fraud risk flag. All three outputs must be explainable and traceable back to the client's own transaction history — this traceability is treated as a defining condition of success, not an optional refinement, since it is what allows the outputs to be trusted and acted on by a loan officer.



## Import the data & Libraries

In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_excel('Loan_approval.xlsx')

## Data Understanding

In [3]:
df.head()

,Branch,Account Holder Name,Account ID,Approval Date,Activation Date,First Repayment Date,Last Payment Date,Product,Loan Amount,Principal Balance,...,Loan usage Business,Completed Loan Cycles (Client),Deposits Balance (Client),Locked Date,Arrears Tolerance Period,Original Account ID,Disbursed Amount,Environmental & Social Risk,Age Band (Client),Nature of business
0,Branch-002,NaN,ACC-8413677F5127,2025-01-05,2025-01-05,2025-02-04,2026-09-27,Product-044,174800,167700,...,BusinessPurpose-007,0,0,NaT,30,NaN,174800,Medium,30-39,Chemist
1,Branch-008,NaN,ACC-02AAD5CBBFD3,2017-05-08,2017-05-08,2017-05-19,2026-09-16,Product-026,3060000,2716200,...,NaN,0,0,NaT,5,NaN,3060000,Medium,30-39,NaN
2,Branch-068,NaN,ACC-C3EA6CE82D4F,2017-07-29,2017-07-28,2017-09-22,2026-05-17,Product-026,4560000,3881300,...,NaN,1,0,NaT,5,NaN,4560000,Medium,40-49,NaN
3,Branch-008,NaN,ACC-BC262E2C2C28,2017-01-13,2016-12-06,2017-01-03,2026-04-18,Product-026,1540000,269600,...,NaN,0,0,NaT,5,NaN,1540000,Medium,NaN,NaN
4,Branch-016,NaN,ACC-B479775F2524,2017-02-08,2017-01-03,2017-01-31,2026-06-15,Product-026,1480000,526100,...,NaN,0,0,NaT,5,NaN,1480000,Medium,NaN,NaN


In [4]:
df.tail()

,Branch,Account Holder Name,Account ID,Approval Date,Activation Date,First Repayment Date,Last Payment Date,Product,Loan Amount,Principal Balance,...,Loan usage Business,Completed Loan Cycles (Client),Deposits Balance (Client),Locked Date,Arrears Tolerance Period,Original Account ID,Disbursed Amount,Environmental & Social Risk,Age Band (Client),Nature of business
23749,Branch-022,NaN,ACC-0352D31E8DF5,2026-07-05,2026-07-05,2026-08-08,2026-09-08,Product-093,5850000,6360700,...,BusinessPurpose-051,0,254300,NaT,3,NaN,6936900,Medium,30-39,Bars/ pubs and clubs / liquor stores
23750,Branch-072,NaN,ACC-5C5734B109E5,2026-06-19,2026-06-19,2026-07-19,2026-08-18,Product-044,1200000,1032800,...,BusinessPurpose-007,0,0,NaT,30,NaN,1200000,Low,20-29,NaN
23751,Branch-023,NaN,ACC-ACAC526857F7,2026-04-12,2026-04-12,2026-05-12,2026-06-30,Product-045,1486800,1339900,...,BusinessPurpose-036,0,1860800,NaT,3,NaN,1635500,Low,40-49,"traditional artefacts, honey and herbs"
23752,Branch-072,NaN,ACC-9C2B6FB9B440,2026-10-27,2026-10-27,2026-11-26,NaT,Product-044,1722700,1722700,...,BusinessPurpose-007,0,0,NaT,30,NaN,1722700,NaN,20-29,NaN
23753,Branch-022,NaN,ACC-C6191020DA44,2026-06-23,2026-06-23,2026-07-26,NaT,Product-048,466200,512800,...,BusinessPurpose-009,14,1693700,NaT,3,NaN,512800,NaN,20-29,Boutiques


In [5]:
df.describe()

,Account Holder Name,Approval Date,Activation Date,First Repayment Date,Last Payment Date,Loan Amount,Principal Balance,Interest Balance,Fee Balance,Interest Accrued,...,Last Payment Amount,Expected Maturity Date,Days Late,Created (Client),Interest Rate,Completed Loan Cycles (Client),Deposits Balance (Client),Locked Date,Arrears Tolerance Period,Disbursed Amount
count,0.0,23754,23754,23754,22025,2.375400e+04,2.375400e+04,2.375400e+04,2.375400e+04,2.375400e+04,...,2.375400e+04,23754,23754.000000,23754,23754.000000,23754.000000,2.375400e+04,644,23754.000000,2.375400e+04
mean,NaN,2025-01-04 00:18:33.008335360,2025-01-10 13:29:21.353877504,2025-02-10 10:58:13.609497600,2026-07-30 08:43:26.056753920,3.257708e+06,2.735155e+06,6.643980e+04,1.655995e+04,5.405495e+04,...,1.693383e+05,2029-09-07 12:36:00.545592320,47.785720,2022-06-26 20:49:17.110381312,7.052780,1.746906,1.016961e+05,2026-02-27 23:55:31.677018624,22.129410,3.492660e+06
min,NaN,2012-07-07 00:00:00,2012-07-07 00:00:00,2012-07-28 00:00:00,2024-06-21 00:00:00,5.000000e+03,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,2015-07-28 00:00:00,0.000000,2010-07-15 00:00:00,0.000000,0.000000,0.000000e+00,2022-03-11 00:00:00,0.000000,5.000000e+03
25%,NaN,2024-04-28 06:00:00,2024-04-29 00:00:00,2024-05-29 06:00:00,2026-06-17 00:00:00,1.000000e+06,7.148000e+05,0.000000e+00,0.000000e+00,5.400000e+03,...,1.980000e+04,2027-05-06 00:00:00,0.000000,2020-04-22 00:00:00,3.500000,0.000000,0.000000e+00,2025-12-28 00:00:00,5.000000,1.048800e+06
50%,NaN,2025-09-06 00:00:00,2025-09-21 00:00:00,2025-10-21 00:00:00,2026-08-02 00:00:00,2.082000e+06,1.708850e+06,0.000000e+00,0.000000e+00,2.060000e+04,...,7.850000e+04,2029-03-09 00:00:00,0.000000,2024-04-08 00:00:00,3.500000,1.000000,0.000000e+00,2026-03-22 00:00:00,30.000000,2.236000e+06
75%,NaN,2026-04-04 00:00:00,2026-04-11 00:00:00,2026-05-12 00:00:00,2026-09-17 00:00:00,4.470000e+06,3.671875e+06,2.160000e+04,2.000000e+02,6.010000e+04,...,1.810000e+05,2031-12-22 00:00:00,9.000000,2025-10-06 00:00:00,3.500000,2.000000,2.567500e+04,2026-06-06 00:00:00,30.000000,4.673450e+06
max,NaN,2026-11-16 00:00:00,2026-11-16 00:00:00,2026-12-16 00:00:00,2026-11-17 00:00:00,4.000000e+07,4.612400e+07,1.418350e+07,1.852950e+07,6.383200e+06,...,1.758160e+07,2034-10-06 00:00:00,4828.000000,2026-11-15 00:00:00,92.470000,40.000000,3.184120e+07,2026-11-08 00:00:00,30.000000,4.612400e+07
std,NaN,NaN,NaN,NaN,NaN,3.542596e+06,3.172838e+06,4.091752e+05,1.611298e+05,1.357705e+05,...,3.642956e+05,NaN,295.347223,NaN,13.441117,2.360336,5.495872e+05,NaN,11.879537,3.919352e+06


In [6]:
df.mean(numeric_only=True)

Account Holder Name                        NaN
Loan Amount                       3.257708e+06
Principal Balance                 2.735155e+06
Interest Balance                  6.643980e+04
Fee Balance                       1.655995e+04
Interest Accrued                  5.405495e+04
Penalty Balance                   2.472127e+03
Days In Arrears                   4.357127e+01
Account Holder ID                          NaN
Principal Due                     1.420166e+05
Interest Due                      6.511200e+04
Fees Due                          1.517882e+04
Penalty Due                       2.472127e+03
Total Due                         2.247801e+05
Principal Paid                    7.575043e+05
Interest Paid                     1.372064e+06
Fees Paid                         2.898453e+05
Total Paid                        2.436880e+06
Number of Installments            5.655186e+01
Last Payment Amount               1.693383e+05
Days Late                         4.778572e+01
Interest Rate

In [7]:
df.var(numeric_only=True)

Account Holder Name                        NaN
Loan Amount                       1.254999e+13
Principal Balance                 1.006690e+13
Interest Balance                  1.674243e+11
Fee Balance                       2.596281e+10
Interest Accrued                  1.843362e+10
Penalty Balance                   9.457016e+08
Days In Arrears                   9.110614e+04
Account Holder ID                          NaN
Principal Due                     5.733680e+11
Interest Due                      1.669204e+11
Fees Due                          1.038681e+10
Penalty Due                       9.457016e+08
Total Due                         1.214846e+12
Principal Paid                    2.949332e+12
Interest Paid                     3.458078e+12
Fees Paid                         2.187879e+11
Total Paid                        1.129534e+13
Number of Installments            1.337836e+03
Last Payment Amount               1.327113e+11
Days Late                         8.722998e+04
Interest Rate

## Checking Missing Values

In [8]:
missing =df.isna().sum()
print(missing)

Branch                                0
Account Holder Name               23754
Account ID                            0
Approval Date                         0
Activation Date                       0
First Repayment Date                  0
Last Payment Date                  1729
Product                               0
Loan Amount                           0
Principal Balance                     0
Interest Balance                      0
Fee Balance                           0
Interest Accrued                      0
Penalty Balance                       0
Days In Arrears                       0
Account Holder ID                 23754
Total Balance                     23387
Principal Due                         0
Interest Due                          0
Fees Due                              0
Penalty Due                           0
Total Due                             0
Principal Paid                        0
Interest Paid                         0
Fees Paid                             0


In [9]:
## check original ID missing values
print(df['Original Account ID'].dropna())

41       ACC-AEB8B58C5651
104      ACC-631DCEC79FB2
107      ACC-66A0B4364D6B
138      ACC-2304A3E1BF88
139      ACC-5B79E6717B4B
145      ACC-DCD334167AE1
185      ACC-9F33F69CD8AB
186      ACC-1A14C57CAFBE
192      ACC-30806A20F510
1772     ACC-D1A3F6D72B27
2150     ACC-D57FE9EDA355
2309     ACC-F22C83423E2A
3570     ACC-B678EBE11425
4101     ACC-117E3877D5C7
4897     ACC-7020F544E2DB
5341     ACC-095C70DF34BE
6595     ACC-75BBF6FB6031
6877     ACC-EDA4738718A5
7207     ACC-8D16EA6F3CBF
7737     ACC-F5DDDFB9C214
9368     ACC-A4F145E9CB01
9610     ACC-15632C0C03CC
11209    ACC-1BEB74ED98B7
11210    ACC-6B606F193857
11748    ACC-791054334C8D
12125    ACC-69376CB3628E
12784    ACC-940782B940CD
13305    ACC-B3FBEED0219A
14923    ACC-39AB9833590D
14971    ACC-311C6C069E57
14972    ACC-478694F7FE7A
14973    ACC-B01C4EC89411
16487    ACC-FFF208AB09C2
16560    ACC-2DB7566ED4FC
16752    ACC-A00EA6E31773
16892    ACC-978E66CC04F0
17069    ACC-037B50D0C3AD
17070    ACC-EB15325E30A9
17893    ACC

In [10]:
## checking if original account ID and account ID are the same
print(df[['Original Account ID', 'Account ID']].head(109))

    Original Account ID        Account ID
0                   NaN  ACC-8413677F5127
1                   NaN  ACC-02AAD5CBBFD3
2                   NaN  ACC-C3EA6CE82D4F
3                   NaN  ACC-BC262E2C2C28
4                   NaN  ACC-B479775F2524
..                  ...               ...
104    ACC-631DCEC79FB2  ACC-BFB26E3D0159
105                 NaN  ACC-604AAC053460
106                 NaN  ACC-7AF3E2704992
107    ACC-66A0B4364D6B  ACC-67E9AEAD42F3
108                 NaN  ACC-12D8F1DF9422

[109 rows x 2 columns]


In [ ]:
## drop the column
df = df.drop(columns=['Original Account ID'], axis=1)

In [12]:
df.columns

Index(['Branch', 'Account Holder Name', 'Account ID', 'Approval Date',
       'Activation Date', 'First Repayment Date', 'Last Payment Date',
       'Product', 'Loan Amount', 'Principal Balance', 'Interest Balance',
       'Fee Balance', 'Interest Accrued', 'Penalty Balance', 'Days In Arrears',
       'Account Holder ID', 'Total Balance', 'Principal Due', 'Interest Due',
       'Fees Due', 'Penalty Due', 'Total Due', 'Principal Paid',
       'Interest Paid', 'Fees Paid', 'Total Paid', 'Number of Installments',
       'Account State', 'Account Sub-State', 'Last Payment Amount',
       'Expected Maturity Date', 'Gender (Client)', 'Loan Usage',
       'Was Refinanced', 'Was Rescheduled', 'Days Late', 'Risk Level',
       'Created (Client)', 'Interest Rate', 'Loan usage Business',
       'Completed Loan Cycles (Client)', 'Deposits Balance (Client)',
       'Locked Date', 'Arrears Tolerance Period', 'Disbursed Amount',
       'Environmental & Social Risk', 'Age Band (Client)',
       'Natur

In [13]:
## check total balance missing values
print(df['Total Balance'].dropna())

0          170,293.99
1        6,287,585.28
10         5501209.91
37       6,903,397.97
82         3709502.62
             ...     
23532    3,513,873.66
23575      294,355.98
23610      409,893.57
23615    3,577,678.04
23737    3,918,927.91
Name: Total Balance, Length: 367, dtype: object


In [14]:
## total balance column
df = df.drop(columns=['Total Balance'])
df['Total Balance'] = (
    df['Principal Balance'] + df['Interest Balance']
    + df['Fee Balance'] + df['Penalty Balance']
)

In [15]:
df = df.drop(columns=['Account Holder Name'])
print(df.isna().sum())

Branch                                0
Account ID                            0
Approval Date                         0
Activation Date                       0
First Repayment Date                  0
Last Payment Date                  1729
Product                               0
Loan Amount                           0
Principal Balance                     0
Interest Balance                      0
Fee Balance                           0
Interest Accrued                      0
Penalty Balance                       0
Days In Arrears                       0
Account Holder ID                 23754
Principal Due                         0
Interest Due                          0
Fees Due                              0
Penalty Due                           0
Total Due                             0
Principal Paid                        0
Interest Paid                         0
Fees Paid                             0
Total Paid                            0
Number of Installments                0


In [16]:
print(df['Account Holder ID'].dropna())

Series([], Name: Account Holder ID, dtype: float64)


In [17]:
## drop account holder ID column
df = df.drop(columns=['Account Holder ID'])

In [ ]:
## check if the missing values in 'Account Sub-State' and 'Locked Date' are the same
sub_null = df['Account Sub-State'].isna()
locked_null = df['Locked Date'].isna()
print((sub_null == locked_null).all()) 

True


In [22]:
df['Account Sub-State'] = df['Account Sub-State'].fillna('Active')

In [24]:
df = df.drop(columns=['Locked Date'])
df.isna().sum()

Branch                               0
Account ID                           0
Approval Date                        0
Activation Date                      0
First Repayment Date                 0
Last Payment Date                 1729
Product                              0
Loan Amount                          0
Principal Balance                    0
Interest Balance                     0
Fee Balance                          0
Interest Accrued                     0
Penalty Balance                      0
Days In Arrears                      0
Principal Due                        0
Interest Due                         0
Fees Due                             0
Penalty Due                          0
Total Due                            0
Principal Paid                       0
Interest Paid                        0
Fees Paid                            0
Total Paid                           0
Number of Installments               0
Account State                        0
Account Sub-State        

In [26]:
## handling last payment date missing values
df['Has Made Last Payment'] = df['Last Payment Date'].notna()
df.columns[-1]

'Has Made Last Payment'

In [27]:
df['Nature of business'].value_counts()

Nature of business
Cooking gas and oil fuels                          2538
Bookshops                                          2056
Agri-business (dairy, meat, poultry, fish)         2012
Bakeries                                           1557
Chemist                                            1252
Agri-business (fruits)                              777
Bars/ pubs and clubs / liquor stores                757
Baby day care services                              421
Woodwork & timber                                   366
Boutiques                                           311
Auto spare parts                                    302
Firewood/ charcoa                                   245
Retail Shops                                        239
traditional artefacts, honey and herbs              221
Health care services                                213
Fuel stations                                       156
Leather & skin tanning                              143
Security services            

In [33]:
df['Nature of business'] = df['Nature of business'].fillna('Unknown')
df['Nature of business'].dropna(inplace=True)
df.isna().sum()

Branch                               0
Account ID                           0
Approval Date                        0
Activation Date                      0
First Repayment Date                 0
Last Payment Date                 1729
Product                              0
Loan Amount                          0
Principal Balance                    0
Interest Balance                     0
Fee Balance                          0
Interest Accrued                     0
Penalty Balance                      0
Days In Arrears                      0
Principal Due                        0
Interest Due                         0
Fees Due                             0
Penalty Due                          0
Total Due                            0
Principal Paid                       0
Interest Paid                        0
Fees Paid                            0
Total Paid                           0
Number of Installments               0
Account State                        0
Account Sub-State        

In [35]:
df['Age Band (Client)'].value_counts()

Age Band (Client)
30-39    9749
40-49    6156
50-59    4143
20-29    3368
60-69     213
<18        69
10-19       6
70+         3
Name: count, dtype: int64

In [ ]:
## filling missing values in age band with mode
df['Age Band (Client)'] = df['Age Band (Client)'].fillna(df['Age Band (Client)'].mode()[0])
df.isna().sum()

Branch                               0
Account ID                           0
Approval Date                        0
Activation Date                      0
First Repayment Date                 0
Last Payment Date                 1729
Product                              0
Loan Amount                          0
Principal Balance                    0
Interest Balance                     0
Fee Balance                          0
Interest Accrued                     0
Penalty Balance                      0
Days In Arrears                      0
Principal Due                        0
Interest Due                         0
Fees Due                             0
Penalty Due                          0
Total Due                            0
Principal Paid                       0
Interest Paid                        0
Fees Paid                            0
Total Paid                           0
Number of Installments               0
Account State                        0
Account Sub-State        

In [38]:
df['Has Made Last Payment'].value_counts()

Has Made Last Payment
True     22025
False     1729
Name: count, dtype: int64

In [39]:
## handling loan usage 
df['Loan Usage'].value_counts()

Loan Usage
LoanPurpose-003    15906
LoanPurpose-004     3132
LoanPurpose-001     1739
LoanPurpose-007     1648
LoanPurpose-006      372
LoanPurpose-005      331
LoanPurpose-002      204
LoanPurpose-008       42
Name: count, dtype: int64

In [41]:
##check extra loan usage business missing values 
extra_missing = df[df['Loan Usage'].notna() & df['Loan usage Business'].isna()]
print(len(extra_missing))

7


In [42]:
print(extra_missing[["Loan Usage", "Loan usage Business"]].head(10))

            Loan Usage Loan usage Business
2115   LoanPurpose-005                 NaN
3681   LoanPurpose-005                 NaN
3809   LoanPurpose-005                 NaN
7938   LoanPurpose-005                 NaN
9763   LoanPurpose-004                 NaN
15353  LoanPurpose-003                 NaN
23548  LoanPurpose-004                 NaN


In [44]:
## filling loan usage business and loan usage with mode
df['Loan usage Business'] = df['Loan usage Business'].fillna(df['Loan usage Business'].mode()[0])
df['Loan Usage'] = df['Loan Usage'].fillna(df['Loan Usage'].mode()[0])
df.isna().sum()

Branch                               0
Account ID                           0
Approval Date                        0
Activation Date                      0
First Repayment Date                 0
Last Payment Date                 1729
Product                              0
Loan Amount                          0
Principal Balance                    0
Interest Balance                     0
Fee Balance                          0
Interest Accrued                     0
Penalty Balance                      0
Days In Arrears                      0
Principal Due                        0
Interest Due                         0
Fees Due                             0
Penalty Due                          0
Total Due                            0
Principal Paid                       0
Interest Paid                        0
Fees Paid                            0
Total Paid                           0
Number of Installments               0
Account State                        0
Account Sub-State        

In [46]:
## environmental & social risk score missing values
df['Environmental & Social Risk'] = df['Environmental & Social Risk'].fillna('Not Assessed')
df.isna().sum()

Branch                               0
Account ID                           0
Approval Date                        0
Activation Date                      0
First Repayment Date                 0
Last Payment Date                 1729
Product                              0
Loan Amount                          0
Principal Balance                    0
Interest Balance                     0
Fee Balance                          0
Interest Accrued                     0
Penalty Balance                      0
Days In Arrears                      0
Principal Due                        0
Interest Due                         0
Fees Due                             0
Penalty Due                          0
Total Due                            0
Principal Paid                       0
Interest Paid                        0
Fees Paid                            0
Total Paid                           0
Number of Installments               0
Account State                        0
Account Sub-State        

In [47]:
df.shape

(23754, 46)

In [48]:
df.info

<bound method DataFrame.info of            Branch        Account ID Approval Date Activation Date  \
0      Branch-002  ACC-8413677F5127    2025-01-05      2025-01-05   
1      Branch-008  ACC-02AAD5CBBFD3    2017-05-08      2017-05-08   
2      Branch-068  ACC-C3EA6CE82D4F    2017-07-29      2017-07-28   
3      Branch-008  ACC-BC262E2C2C28    2017-01-13      2016-12-06   
4      Branch-016  ACC-B479775F2524    2017-02-08      2017-01-03   
...           ...               ...           ...             ...   
23749  Branch-022  ACC-0352D31E8DF5    2026-07-05      2026-07-05   
23750  Branch-072  ACC-5C5734B109E5    2026-06-19      2026-06-19   
23751  Branch-023  ACC-ACAC526857F7    2026-04-12      2026-04-12   
23752  Branch-072  ACC-9C2B6FB9B440    2026-10-27      2026-10-27   
23753  Branch-022  ACC-C6191020DA44    2026-06-23      2026-06-23   

      First Repayment Date Last Payment Date      Product  Loan Amount  \
0               2025-02-04        2026-09-27  Product-044       1